# A100 DnCNN pretrained fine-tuning

공식 `cszn/KAIR`의 `dncnn_gray_blind.pth`를 시작점으로 사용해 반도체 grayscale 이미지에 fine-tuning합니다.

- 학습 및 모델 선택에는 `train`과 `val`만 사용합니다.
- 테스트 데이터는 best checkpoint가 확정된 다음 셀에서 처음 복사하고, 최종 평가에만 사용합니다.
- 원본 `train_denoising_example.ipynb`와 `test_denoising.ipynb`는 수정하지 않습니다.
- 데이터 복사, 학습, x8 최종 평가를 포함해 A100 런타임 1시간 이내를 목표로 합니다.


## 1. Colab 및 프로젝트 경로 설정


In [ ]:
import time

NOTEBOOK_STARTED_AT = time.monotonic()

from google.colab import drive
drive.mount("/content/drive")

import json
import math
import os
import random
import shutil
import sys
import time
import urllib.request
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# 원본 예제 노트북과 동일한 기본 경로입니다. Drive 위치가 다르면 이 한 줄만 바꾸세요.
ROOT = Path("/content/drive/MyDrive/Data scientist/팀프로젝트/5주차")
CODE_DIR = ROOT / "code_denoising"
DATA_ROOT = ROOT / "dataset"
MODULE_PATH = CODE_DIR / "a100_denoising.py"

if not MODULE_PATH.exists():
    raise FileNotFoundError(
        f"{MODULE_PATH}가 없습니다. ROOT를 현재 프로젝트 폴더로 수정하세요."
    )

sys.path.insert(0, str(CODE_DIR))
from a100_denoising import (
    CleanDenoisingDataset,
    DataKey,
    DnCNN,
    ExponentialMovingAverage,
    NOISE_RANGES,
    PairedDenoisingDataset,
    TestTimeDenoiser,
    TimeBudget,
    calculate_psnr,
    calculate_ssim,
    discover_clean_files,
    evaluate_model,
    load_finetuned_checkpoint,
    load_pretrained_weights,
    save_checkpoint,
    train_one_epoch,
    validate_disjoint_splits,
)

print("project:", ROOT)
print("torch:", torch.__version__)


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 아닙니다. Colab 런타임을 A100 GPU로 변경하세요.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)
if "A100" not in gpu_name.upper():
    print("경고: A100이 아닙니다. 시간 제한을 위해 batch size를 낮춰야 할 수 있습니다.")

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
TEST_RUN_DIR = ROOT / "logs_denoising_a100" / RUN_ID
TEST_RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run dir:", TEST_RUN_DIR)


## 2. train/val만 `/content`로 복사

이 단계에서는 테스트 디렉터리를 읽거나 복사하지 않습니다. Google Drive I/O가 학습 병목이 되지 않도록 clean train/val만 로컬 SSD로 복사합니다.


In [ ]:
import zipfile
from pathlib import PurePosixPath

LOCAL_DATA_ROOT = Path("/content/denoising_dataset")
LOCAL_TRAIN_DIR = LOCAL_DATA_ROOT / "train"
LOCAL_VAL_DIR = LOCAL_DATA_ROOT / "val"
LOCAL_ARCHIVE = Path("/content/dataset.zip")

archive_candidates = [
    ROOT / "dataset.zip",
    DATA_ROOT / "dataset.zip",
]
ARCHIVE_SOURCE = next(
    (path for path in archive_candidates if path.exists()),
    None,
)

if ARCHIVE_SOURCE is None:
    raise FileNotFoundError(
        f"dataset.zip을 찾을 수 없습니다: {archive_candidates}"
    )


def copy_archive_with_progress(source: Path, destination: Path) -> None:
    if (
        destination.exists()
        and destination.stat().st_size == source.stat().st_size
    ):
        print("로컬 ZIP 재사용:", destination)
        return

    temporary = destination.with_suffix(".zip.part")
    temporary.unlink(missing_ok=True)

    chunk_size = 16 * 1024 * 1024
    total_size = source.stat().st_size

    with (
        source.open("rb") as source_file,
        temporary.open("wb") as destination_file,
        tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            desc="dataset.zip 복사",
        ) as progress,
    ):
        while True:
            chunk = source_file.read(chunk_size)
            if not chunk:
                break
            destination_file.write(chunk)
            progress.update(len(chunk))

    temporary.replace(destination)

    if destination.stat().st_size != total_size:
        raise IOError("dataset.zip 복사가 완전하지 않습니다.")


def extract_selected_splits(
    archive_path: Path,
    destination_root: Path,
    splits: tuple[str, ...],
) -> None:
    for split in splits:
        destination = destination_root / split
        if destination.exists():
            shutil.rmtree(destination)
        destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path) as archive:
        selected = []

        for info in archive.infolist():
            if info.is_dir():
                continue

            normalized = info.filename.replace("\\", "/")
            parts = PurePosixPath(normalized).parts

            if ".." in parts:
                raise ValueError(f"안전하지 않은 ZIP 경로: {info.filename}")

            matched_split = next(
                (split for split in splits if split in parts),
                None,
            )
            if matched_split is None:
                continue

            split_index = parts.index(matched_split)
            relative_parts = list(parts[split_index + 1 :])

            while (
                relative_parts
                and relative_parts[0] == matched_split
            ):
                relative_parts.pop(0)

            if not relative_parts:
                continue
            if Path(relative_parts[-1]).suffix.lower() != ".npy":
                continue

            selected.append(
                (info, matched_split, relative_parts)
            )

        if not selected:
            raise FileNotFoundError(
                f"ZIP에서 {splits} split을 찾지 못했습니다."
            )

        for info, split, relative_parts in tqdm(
            selected,
            desc="train/val 압축 해제",
        ):
            output_path = (
                destination_root
                / split
                / Path(*relative_parts)
            )
            output_path.parent.mkdir(parents=True, exist_ok=True)

            with (
                archive.open(info) as source_file,
                output_path.open("wb") as output_file,
            ):
                shutil.copyfileobj(
                    source_file,
                    output_file,
                    length=16 * 1024 * 1024,
                )


# freeze 이후 test 데이터 100장을 복사할 때 기존 셀이 사용합니다.
def copy_split(source: Path, destination: Path) -> None:
    source_files = sorted(source.glob("*.npy"))
    if not source_files:
        raise FileNotFoundError(f"데이터가 없습니다: {source}")

    if destination.exists():
        shutil.rmtree(destination)

    shutil.copytree(source, destination)
    print(f"{source.name}: {len(source_files)} files -> {destination}")


copy_archive_with_progress(ARCHIVE_SOURCE, LOCAL_ARCHIVE)

# 학습 전에는 train과 val만 압축 해제합니다.
extract_selected_splits(
    LOCAL_ARCHIVE,
    LOCAL_DATA_ROOT,
    ("train", "val"),
)

print("train:", len(list(LOCAL_TRAIN_DIR.glob("*.npy"))))
print("val:", len(list(LOCAL_VAL_DIR.glob("*.npy"))))

In [ ]:
TRAIN_BATCH = 64
PATCH_SIZE = 128
NUM_WORKERS = min(8, os.cpu_count() or 2)

train_files = discover_clean_files([LOCAL_TRAIN_DIR])
validation_files = discover_clean_files([LOCAL_VAL_DIR])
validate_disjoint_splits(train_files, validation_files)

train_dataset = CleanDenoisingDataset(
    train_files,
    patch_size=PATCH_SIZE,
    training=True,
    base_seed=SEED,
    identity_probability=0.08,
)
validation_dataset = CleanDenoisingDataset(
    validation_files,
    patch_size=None,
    training=False,
    base_seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=min(4, NUM_WORKERS),
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

print("train:", len(train_dataset), "validation:", len(validation_dataset))
print("steps per epoch:", len(train_loader))


## 3. 공식 pretrained grayscale-blind DnCNN 로드


In [ ]:
PRETRAINED_URL = (
    "https://github.com/cszn/KAIR/releases/download/v1.0/"
    "dncnn_gray_blind.pth"
)
PRETRAINED_PATH = Path("/content/dncnn_gray_blind.pth")
MODEL_CONFIG = {"depth": 20, "channels": 1, "features": 64}

if not PRETRAINED_PATH.exists() or PRETRAINED_PATH.stat().st_size < 2_000_000:
    print("Downloading official dncnn_gray_blind.pth ...")
    urllib.request.urlretrieve(PRETRAINED_URL, PRETRAINED_PATH)

model = DnCNN(**MODEL_CONFIG)
load_pretrained_weights(model, PRETRAINED_PATH)
model = model.to(device=device, memory_format=torch.channels_last)

with torch.no_grad():
    sanity = model(torch.rand(1, 1, 32, 32, device=device))
assert sanity.shape == (1, 1, 32, 32) and torch.isfinite(sanity).all()
print("pretrained strict load OK | params:", sum(p.numel() for p in model.parameters()))


## 4. Fine-tuning

- 공식 pretrained weight에서 모든 layer를 fine-tuning합니다.
- MSE를 중심으로 작은 SSIM 항을 더해 PSNR과 구조 보존을 함께 최적화합니다.
- EMA weight와 validation PSNR로 best checkpoint를 결정합니다.
- 학습 시간은 45분에서 중단하여 checkpoint 저장과 최종 평가 시간을 남깁니다.


In [ ]:
MAX_EPOCHS = 40
MIN_EPOCHS = 12
EARLY_STOP_PATIENCE = 8
MAX_TOTAL_MINUTES = 55.0
FINAL_RESERVE_MINUTES = 8.0
SSIM_WEIGHT = 0.02
MAX_LR = 2e-4

optimizer = torch.optim.AdamW(
    model.parameters(), lr=MAX_LR / 10.0, betas=(0.9, 0.99), weight_decay=1e-6
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    total_steps=MAX_EPOCHS * len(train_loader),
    pct_start=0.1,
    anneal_strategy="cos",
    div_factor=10.0,
    final_div_factor=100.0,
)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=True)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=True)

ema = ExponentialMovingAverage(model, decay=0.999)
BEST_CHECKPOINT = TEST_RUN_DIR / "checkpoint_best_a100.ckpt"
HISTORY_PATH = TEST_RUN_DIR / "training_history.json"

best_psnr = -math.inf
bad_epochs = 0
history = []
budget = TimeBudget(
    MAX_TOTAL_MINUTES,
    FINAL_RESERVE_MINUTES,
    NOTEBOOK_STARTED_AT,
)


In [ ]:
for epoch in range(1, MAX_EPOCHS + 1):
    training_deadline = (
        budget.start_time
        + (budget.max_minutes - budget.reserve_minutes) * 60.0
    )
    expected_epoch_seconds = (
        max(record["seconds"] for record in history) * 1.25
        if history
        else 0.0
    )
    if budget.expired() or (
        expected_epoch_seconds > 0
        and time.monotonic() + expected_epoch_seconds >= training_deadline
    ):
        print("final evaluation reserve reached; training stops before a new epoch")
        break

    epoch_start = time.monotonic()
    train_loss = train_one_epoch(
        model,
        tqdm(train_loader, desc=f"train {epoch:02d}", leave=False),
        optimizer,
        device=device,
        ssim_weight=SSIM_WEIGHT,
        scaler=scaler,
        scheduler=scheduler,
        ema=ema,
        gradient_clip=1.0,
    )
    val_psnr, val_ssim = evaluate_model(
        ema.model,
        tqdm(validation_loader, desc=f"valid {epoch:02d}", leave=False),
        device,
        clamp=True,
    )
    elapsed = time.monotonic() - epoch_start
    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_psnr": val_psnr,
        "val_ssim": val_ssim,
        "lr": optimizer.param_groups[0]["lr"],
        "seconds": elapsed,
    }
    history.append(record)
    HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding="utf-8")
    print(
        f"epoch {epoch:02d} | loss {train_loss:.5f} | "
        f"val PSNR {val_psnr:.3f} | SSIM {val_ssim:.4f} | {elapsed:.1f}s"
    )

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        bad_epochs = 0
        save_checkpoint(
            BEST_CHECKPOINT,
            ema.model,
            MODEL_CONFIG,
            epoch=epoch,
            val_psnr=val_psnr,
            val_ssim=val_ssim,
        )
        print("  best checkpoint updated")
    else:
        bad_epochs += 1

    if budget.expired():
        print("시간 제한에 도달하여 학습을 종료합니다.")
        break
    if epoch >= MIN_EPOCHS and bad_epochs >= EARLY_STOP_PATIENCE:
        print("validation PSNR early stopping")
        break

if not BEST_CHECKPOINT.exists():
    raise RuntimeError("best checkpoint가 생성되지 않았습니다.")
print("best checkpoint:", BEST_CHECKPOINT)


In [ ]:
# 이 셀에서 학습 및 모델 선택을 완전히 종료합니다.
frozen_model, best_metadata = load_finetuned_checkpoint(BEST_CHECKPOINT, device)
frozen_model.requires_grad_(False).eval()
for parameter in frozen_model.parameters():
    assert not parameter.requires_grad

print("FROZEN BEST MODEL")
print(best_metadata)
print("이후 test 결과는 checkpoint 선택이나 hyperparameter 변경에 사용하지 않습니다.")


## 5. Validation-only impulse 전처리 ablation

Salt-and-pepper 후보 픽셀을 median으로 교정하는 전처리를 validation에서만 비교합니다. PSNR이 개선되고 SSIM 손실이 없을 때만 최종 inference 경로에 자동 채택합니다.

In [ ]:
def impulse_correction_v2(
    x: Tensor,
    thr: float = 0.1,
    gate: float = 0.005,
) -> tuple[Tensor, Tensor]:
    """
    이미지별로 salt-and-pepper 가능성을 판단한 뒤,
    확실한 impulse 픽셀만 3x3 median으로 교체합니다.

    입력:
        [C, H, W] 또는 [B, C, H, W]
    """
    if x.dim() not in (3, 4):
        raise ValueError(
            f"expected 3D or 4D tensor, got {x.shape}"
        )

    squeeze_batch = x.dim() == 3
    x4 = x.unsqueeze(0) if squeeze_batch else x

    # 배치 전체가 아닌 이미지별 최댓값
    image_max = x4.amax(
        dim=(-2, -1),
        keepdim=True,
    )

    extreme = (x4 == 0) | (x4 == image_max)

    # 이미지별 extreme pixel 비율
    extreme_ratio = extreme.float().mean(
        dim=(1, 2, 3),
        keepdim=True,
    )

    padded = functional.pad(
        x4,
        (1, 1, 1, 1),
        mode="reflect",
    )

    patches = (
        padded
        .unfold(2, 3, 1)
        .unfold(3, 3, 1)
    )

    median = (
        patches
        .contiguous()
        .view(*patches.shape[:4], 9)
        .median(dim=-1)
        .values
    )

    impulse_candidate = (
        extreme
        & ((x4 - median).abs() > thr)
    )

    gate_passed = extreme_ratio >= gate
    mask = impulse_candidate & gate_passed

    corrected = torch.where(mask, median, x4)

    if squeeze_batch:
        return corrected[0], mask[0]

    return corrected, mask


class ImpulsePreprocessedDenoiser(nn.Module):
    """
    impulse correction을 한 번 적용한 뒤
    기존 DnCNN+x8 test_network를 호출합니다.
    """

    def __init__(
        self,
        denoiser: nn.Module,
        thr: float = 0.1,
        gate: float = 0.005,
    ):
        super().__init__()
        self.denoiser = denoiser
        self.thr = thr
        self.gate = gate

    def forward(self, noisy: Tensor) -> Tensor:
        corrected, _ = impulse_correction_v2(
            noisy,
            thr=self.thr,
            gate=self.gate,
        )
        return self.denoiser(corrected)


# frozen best checkpoint에 기존 x8 inference를 결합합니다.
dncnn_only_network = TestTimeDenoiser(
    frozen_model,
    use_x8=True,
    clamp=True,
).to(device).eval()
dncnn_only_network.requires_grad_(False)

impulse_test_network = ImpulsePreprocessedDenoiser(
    dncnn_only_network,
    thr=0.1,
    gate=0.005,
).to(device).eval()

impulse_test_network.requires_grad_(False)


# test가 아닌 validation으로만 적용 여부 결정
print("Validation A/B 평가 중...")

dncnn_val_psnr, dncnn_val_ssim = evaluate_model(
    dncnn_only_network,
    validation_loader,
    device,
    clamp=True,
)

impulse_val_psnr, impulse_val_ssim = evaluate_model(
    impulse_test_network,
    validation_loader,
    device,
    clamp=True,
)

print()
print("[기존 DnCNN]")
print(f"PSNR: {dncnn_val_psnr:.4f}")
print(f"SSIM: {dncnn_val_ssim:.6f}")

print()
print("[Impulse correction + DnCNN]")
print(f"PSNR: {impulse_val_psnr:.4f}")
print(f"SSIM: {impulse_val_ssim:.6f}")
print()
print("[변화]")
print(
    f"PSNR: {impulse_val_psnr - dncnn_val_psnr:+.4f} dB"
)
print(
    f"SSIM: {impulse_val_ssim - dncnn_val_ssim:+.6f}"
)


# PSNR이 개선되고 SSIM 손실이 사실상 없을 때만 자동 채택
use_impulse_correction = (
    impulse_val_psnr > dncnn_val_psnr
    and impulse_val_ssim >= dncnn_val_ssim - 0.0001
)

if use_impulse_correction:
    test_network = impulse_test_network
    selected_method = "impulse_correction_v2 + DnCNN x8"
else:
    test_network = dncnn_only_network
    selected_method = "DnCNN x8"

print()
print("선택된 최종 방법:", selected_method)


# 선택 근거 저장
impulse_decision = {
    "selection_data": "validation_only",
    "threshold": 0.1,
    "gate": 0.005,
    "dncnn_val_psnr": float(dncnn_val_psnr),
    "dncnn_val_ssim": float(dncnn_val_ssim),
    "impulse_val_psnr": float(impulse_val_psnr),
    "impulse_val_ssim": float(impulse_val_ssim),
    "psnr_change": float(
        impulse_val_psnr - dncnn_val_psnr
    ),
    "ssim_change": float(
        impulse_val_ssim - dncnn_val_ssim
    ),
    "use_impulse_correction": use_impulse_correction,
    "selected_method": selected_method,
}

decision_path = (
    Path(TEST_RUN_DIR)
    / "impulse_correction_validation.json"
)

decision_path.write_text(
    json.dumps(
        impulse_decision,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("결정 기록 저장:", decision_path)

## 6. Frozen model 최종 평가 준비

모델과 전처리 선택이 끝난 뒤에만 test 데이터를 로드합니다.

In [ ]:
LOCAL_LABEL_DIR = LOCAL_DATA_ROOT / "test_label"
LOCAL_NOISY_DIR = LOCAL_DATA_ROOT / "test_noise_only"

label_source = DATA_ROOT / "test_label"
noisy_source = DATA_ROOT / "test_noise_only"
if not list(noisy_source.glob("*.npy")):
    noisy_source = noisy_source / "test_noise_only"

copy_split(label_source, LOCAL_LABEL_DIR)
copy_split(noisy_source, LOCAL_NOISY_DIR)
meta_source = noisy_source / "noise_meta.json"
if meta_source.exists():
    shutil.copy2(meta_source, LOCAL_NOISY_DIR / "noise_meta.json")

paired_test_dataset = PairedDenoisingDataset(LOCAL_LABEL_DIR, LOCAL_NOISY_DIR)
test_loader = DataLoader(
    paired_test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

noise_meta_path = LOCAL_NOISY_DIR / "noise_meta.json"
noise_lookup = {}
if noise_meta_path.exists():
    noise_rows = json.loads(noise_meta_path.read_text(encoding="utf-8"))
    noise_lookup = {row["file"]: row["noise_type"] for row in noise_rows}

# 원본 test notebook이 기대하는 최소 config 계약입니다.
from types import SimpleNamespace
config = SimpleNamespace(
    device=device,
    test_dataset=[str(LOCAL_LABEL_DIR)],
    test_noisy_dir=str(LOCAL_NOISY_DIR),
)

# test_denoising.ipynb가 그대로 호출할 drop-in model입니다.
print("final test images:", len(paired_test_dataset))


## 7. 원본 평가 노트북 실행

원본 `test_denoising.ipynb`를 수정하지 않고 같은 namespace에서 실행하여 정량 지표와 필터 baseline을 저장합니다.

In [ ]:
TEST_NOTEBOOK = CODE_DIR / "test_denoising.ipynb"
if not TEST_NOTEBOOK.exists():
    raise FileNotFoundError(TEST_NOTEBOOK)

# Optional PNG preview cell(5)은 앞 단계의 임시 파일에 의존하므로 제외합니다.
# 정량 지표와 baseline에 필요한 원본 셀만 같은 namespace에서 실행합니다.
ORIGINAL_TEST_CODE_CELLS = (3, 7, 8, 10)
original_notebook = json.loads(TEST_NOTEBOOK.read_text(encoding="utf-8"))
for cell_index in ORIGINAL_TEST_CODE_CELLS:
    original_cell = original_notebook["cells"][cell_index]
    if original_cell.get("cell_type") != "code":
        raise RuntimeError(f"expected code cell at index {cell_index}")
    source = "".join(original_cell.get("source", []))
    print(f"running original test cell {cell_index}")
    exec(
        compile(source, f"{TEST_NOTEBOOK}:cell_{cell_index}", "exec"),
        globals(),
    )

print("최종 결과 저장:", TEST_RUN_DIR)


## 8. Noise별 대표 이미지 선정

각 noise에서 DnCNN PSNR·SSIM 중앙값에 가장 가까운 샘플을 대표 이미지로 선정합니다.

In [ ]:
# noise 종류별로 PSNR·SSIM 중앙값에 가장 가까운 대표 이미지 선정
representative_rows = {}

for noise_type in NOISE_RANGES:
    rows = [
        row
        for row in baseline_rows
        if row["noise_type"] == noise_type
    ]

    if not rows:
        continue

    psnr_values = np.asarray(
        [row["psnr_dncnn"] for row in rows],
        dtype=np.float64,
    )
    ssim_values = np.asarray(
        [row["ssim_dncnn"] for row in rows],
        dtype=np.float64,
    )

    median_psnr = float(np.median(psnr_values))
    median_ssim = float(np.median(ssim_values))

    psnr_mad = float(
        np.median(np.abs(psnr_values - median_psnr))
    )
    ssim_mad = float(
        np.median(np.abs(ssim_values - median_ssim))
    )

    psnr_scale = max(psnr_mad, 1e-6)
    ssim_scale = max(ssim_mad, 1e-6)

    def representative_distance(row):
        psnr_distance = (
            (row["psnr_dncnn"] - median_psnr)
            / psnr_scale
        )
        ssim_distance = (
            (row["ssim_dncnn"] - median_ssim)
            / ssim_scale
        )
        return psnr_distance**2 + ssim_distance**2

    representative_rows[noise_type] = min(
        rows,
        key=representative_distance,
    )


selected_names = {
    row["file"]
    for row in representative_rows.values()
}

# 선택된 4개 이미지만 다시 불러와 DnCNN 결과 생성
representative_samples = {}

test_network.eval()

with torch.inference_mode():
    for data in test_loader:
        names = data[DataKey.Name]

        selected_indices = [
            index
            for index, name in enumerate(names)
            if name in selected_names
        ]

        if not selected_indices:
            continue

        label = data[DataKey.Label][selected_indices].to(
            config.device
        )
        noisy = data[DataKey.Noisy][selected_indices].to(
            config.device
        )
        restored = test_network(noisy)

        for batch_index, original_index in enumerate(
            selected_indices
        ):
            name = names[original_index]

            row = next(
                row
                for row in representative_rows.values()
                if row["file"] == name
            )
            noise_type = row["noise_type"]

            representative_samples[noise_type] = {
                "name": name,
                "metrics": row,
                "noisy": (
                    noisy[batch_index]
                    .detach()
                    .cpu()
                    .numpy()
                    .squeeze()
                ),
                "dncnn": (
                    restored[batch_index]
                    .detach()
                    .cpu()
                    .numpy()
                    .squeeze()
                ),
                "label": (
                    label[batch_index]
                    .detach()
                    .cpu()
                    .numpy()
                    .squeeze()
                ),
            }

        if len(representative_samples) == len(
            representative_rows
        ):
            break


# 기존 시각화 코드가 그대로 사용하도록 교체
noise_samples = representative_samples

print("선택된 대표 이미지")
for noise_type, sample in noise_samples.items():
    metrics = sample["metrics"]
    print(
        f"{noise_type:<18} "
        f"{sample['name']} | "
        f"PSNR {metrics['psnr_dncnn']:.2f} | "
        f"SSIM {metrics['ssim_dncnn']:.4f}"
    )

## 9. Noisy / DnCNN / Clean / Error 비교

In [ ]:
ERROR_CMAP = "magma"

row_order = [
    noise_type
    for noise_type in [*NOISE_RANGES.keys(), "unknown"]
    if noise_type in noise_samples
]

if not row_order:
    raise RuntimeError(
        "noise_samples가 비어 있습니다. baseline 평가 셀을 먼저 실행하세요."
    )

fig, axes = plt.subplots(
    len(row_order),
    4,
    figsize=(18, 4.5 * len(row_order)),
    squeeze=False,
)

for row_index, noise_type in enumerate(row_order):
    sample = noise_samples[noise_type]
    noisy = sample["noisy"]
    restored = sample["dncnn"]
    clean = sample["label"]
    metrics = sample["metrics"]

    display_min = float(np.percentile(clean, 1))
    display_max = float(np.percentile(clean, 99))

    absolute_error = np.abs(restored - clean)
    error_max = float(
        max(np.percentile(absolute_error, 99.5), 1e-6)
    )

    panels = [
        (
            noisy,
            "gray",
            display_min,
            display_max,
            "Noisy input\n"
            f"PSNR {metrics['psnr_noisy']:.2f} dB / "
            f"SSIM {metrics['ssim_noisy']:.4f}",
        ),
        (
            restored,
            "gray",
            display_min,
            display_max,
            "DnCNN restored\n"
            f"PSNR {metrics['psnr_dncnn']:.2f} dB / "
            f"SSIM {metrics['ssim_dncnn']:.4f}",
        ),
        (
            clean,
            "gray",
            display_min,
            display_max,
            "Clean original (label)",
        ),
        (
            absolute_error,
            ERROR_CMAP,
            0.0,
            error_max,
            "DnCNN absolute error\n"
            f"mean {absolute_error.mean():.5f} / "
            f"max {absolute_error.max():.5f}",
        ),
    ]

    for column_index, (
        image,
        color_map,
        value_min,
        value_max,
        title,
    ) in enumerate(panels):
        axis = axes[row_index, column_index]

        rendered = axis.imshow(
            image,
            cmap=color_map,
            vmin=value_min,
            vmax=value_max,
        )
        axis.set_title(title, fontsize=10)
        axis.axis("off")

        if column_index == 0:
            axis.set_ylabel(
                f"{noise_type}\n{sample['name']}",
                fontsize=10,
            )
            axis.yaxis.set_visible(True)
            axis.set_yticks([])

        if column_index == 3:
            fig.colorbar(
                rendered,
                ax=axis,
                fraction=0.046,
                pad=0.04,
            )

fig.suptitle(
    "DnCNN Denoising: Noisy → Restored → Clean Original",
    fontsize=15,
    fontweight="bold",
)

fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.subplots_adjust(hspace=0.28)

comparison_path = (
    TEST_RUN_DIR / "dncnn_clean_comparison.png"
)
fig.savefig(
    comparison_path,
    dpi=180,
    bbox_inches="tight",
)

print("saved:", comparison_path)
plt.show()

## 10. 저장 결과와 운영 규칙

- `checkpoint_best_a100.ckpt`: validation PSNR 기준 최고 모델
- `training_history.json`: epoch별 학습·validation 기록
- `impulse_correction_validation.json`: validation-only ablation 결과
- `baseline_metrics.json`: 최종 test 정량 지표
- `baseline_grid.png`: 기존 필터와 DnCNN 비교
- `dncnn_clean_comparison.png`: 대표 샘플 복원 결과

Test 결과를 학습률, epoch, loss, threshold 선택에 다시 사용하지 않습니다.